# Comparação de Modelos para Predição de Churn

## Objetivo

Esta etapa tem como objetivo comparar diferentes algoritmos de classificação para a predição de churn, utilizando o mesmo conjunto de treinamento e teste e as mesmas métricas de avaliação.

Serão comparados inicialmente:

- Regressão Logística
- Random Forest

A comparação utilizará como métrica principal o F1-score da classe positiva (Churn = 1), acompanhada por Precision, Recall, ROC-AUC e matriz de confusão.

Como a análise anterior demonstrou desbalanceamento da variável-alvo, os modelos serão avaliados considerando estratégias adequadas para a classe minoritária.

O objetivo é selecionar um modelo que apresente bom equilíbrio entre a identificação dos clientes com risco de churn e a quantidade de falsos positivos.

In [2]:
#Import de bbliotecas e reprodução do split 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

RANDOM_STATE = 42
TEST_SIZE = 0.20


In [3]:
caminho_dados = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"

dados = pd.read_csv(caminho_dados)

dados.shape

(7043, 21)

In [4]:
dados["TotalCharges"] = pd.to_numeric(
    dados["TotalCharges"],
    errors="coerce"
)

dados["TotalCharges"].isna().sum()


np.int64(11)

In [5]:
dados = dados.dropna(subset=["TotalCharges"]).copy()

dados.shape

(7032, 21)

In [6]:
#separação de features e variável alvo (churn)
X = dados.drop(columns=["customerID", "Churn"]).copy()

y = dados["Churn"].map({
    "No": 0,
    "Yes": 1
})

print("Dimensão de X:", X.shape)
print("\nDistribuição de y:")
print(y.value_counts())
print("\nProporção de churn:")
print(y.value_counts(normalize=True).round(4))

Dimensão de X: (7032, 19)

Distribuição de y:
Churn
0    5163
1    1869
Name: count, dtype: int64

Proporção de churn:
Churn
0    0.7342
1    0.2658
Name: proportion, dtype: float64


In [7]:
#split treino / teste e stratify para manter a mesma proporção churn/não churn 
# entre teste e treino
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

print("\nChurn no treino:")
print(y_train.value_counts(normalize=True).round(4))

print("\nChurn no teste:")
print(y_test.value_counts(normalize=True).round(4))

Treino: (5625, 19)
Teste: (1407, 19)

Churn no treino:
Churn
0    0.7342
1    0.2658
Name: proportion, dtype: float64

Churn no teste:
Churn
0    0.7342
1    0.2658
Name: proportion, dtype: float64


In [8]:
#pre processing / one hot encoder
colunas_numericas = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

colunas_categoricas = [
    coluna for coluna in X_train.columns
    if coluna not in colunas_numericas
]

preprocessamento = ColumnTransformer(
    transformers=[
        (
            "numericas",
            StandardScaler(),
            colunas_numericas
        ),
        (
            "categoricas",
            OneHotEncoder(
                drop="if_binary",
                handle_unknown="ignore"
            ),
            colunas_categoricas
        )
    ]
)


In [9]:
# envio da mesma base de treinamento aos dois modelos

modelo_regressao_logistica = Pipeline(
    steps=[
        ("preprocessamento", preprocessamento),
        (
            "modelo",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=RANDOM_STATE
            )
        )
    ]
)

modelo_random_forest = Pipeline(
    steps=[
        ("preprocessamento", preprocessamento),
        (
            "modelo",
            RandomForestClassifier(
                n_estimators=300,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        )
    ]
)

In [10]:
from sklearn.model_selection import StratifiedKFold, cross_validate

In [11]:
#comparação entre modelos

validacao_cruzada = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

metricas = {
    "f1": "f1",
    "recall": "recall",
    "precision": "precision",
    "roc_auc": "roc_auc"
}

modelos = {
    "Regressão Logística": modelo_regressao_logistica,
    "Random Forest": modelo_random_forest
}

resultados = []

for nome_modelo, modelo in modelos.items():
    resultado_validacao = cross_validate(
        modelo,
        X_train,
        y_train,
        cv=validacao_cruzada,
        scoring=metricas,
        n_jobs=-1
    )

    resultados.append({
        "Modelo": nome_modelo,
        "F1": resultado_validacao["test_f1"].mean(),
        "Recall": resultado_validacao["test_recall"].mean(),
        "Precision": resultado_validacao["test_precision"].mean(),
        "ROC-AUC": resultado_validacao["test_roc_auc"].mean()
    })

comparacao_modelos = pd.DataFrame(resultados).set_index("Modelo")

comparacao_modelos.round(3)

,F1,Recall,Precision,ROC-AUC
Modelo,,,,
Regressão Logística,0.633,0.803,0.522,0.846
Random Forest,0.608,0.656,0.567,0.827


# Comparação inicial dos modelos

Nesta etapa, foram comparados dois algoritmos para predição de churn:

- Regressão Logística
- Random Forest

Para garantir uma comparação justa, ambos utilizaram:

- o mesmo conjunto de treinamento;
- o mesmo pré-processamento das variáveis;
- class_weight="balanced" para lidar com o desbalanceamento da variável-alvo;
- validação cruzada estratificada com 5 folds;
- F1-score da classe de churn como principal métrica de comparação.

# Resultados da validação cruzada

A Regressão Logística apresentou os seguintes resultados:

- F1-score: 0.633
- Recall: 0.803
- Precision: 0.522
- ROC-AUC: 0.846

A Random Forest apresentou:

- F1-score: 0.608
- Recall: 0.656
- Precision: 0.567
- ROC-AUC: 0.827

A Regressão Logística apresentou o melhor desempenho geral, obtendo maior F1-score, Recall e ROC-AUC.

A Random Forest apresentou maior Precision, indicando menor proporção de falsos positivos entre os clientes classificados como churn. Entretanto, seu Recall foi consideravelmente menor, fazendo com que mais clientes que efetivamente apresentam churn deixassem de ser identificados.

Para o problema de retenção de clientes, o Recall possui especial relevância, pois falsos negativos representam clientes em risco de churn que não seriam identificados preventivamente.

# Próxima etapa

Antes de selecionar definitivamente a Regressão Logística, será realizado um ajuste de hiperparâmetros da Random Forest.

O objetivo é verificar se a diferença observada decorre das características do algoritmo ou da configuração inicial utilizada.

O tuning será realizado exclusivamente sobre o conjunto de treinamento, utilizando validação cruzada estratificada e F1-score como critério principal de seleção.

O conjunto de teste permanecerá isolado durante todo esse processo e será utilizado somente após a escolha do modelo final, permitindo uma avaliação mais confiável da capacidade de generalização.

In [12]:
from sklearn.model_selection import RandomizedSearchCV

In [13]:
parametros_random_forest = {
    "modelo__n_estimators": [100, 200, 300, 500],
    "modelo__max_depth": [None, 5, 10, 15, 20],
    "modelo__min_samples_split": [2, 5, 10, 20],
    "modelo__min_samples_leaf": [1, 2, 4, 8, 10],
    "modelo__max_features": ["sqrt", "log2", None]
}

In [14]:
# tentativa de otimização do random forest com 30 combinações aleatórias de hiperparâmetros 
busca_random_forest = RandomizedSearchCV(
    estimator=modelo_random_forest,
    param_distributions=parametros_random_forest,
    n_iter=30,
    scoring="f1",
    cv=validacao_cruzada,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

In [15]:
busca_random_forest.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'modelo__max_depth': [None, 5, ...], 'modelo__max_features': ['sqrt', 'log2', ...], 'modelo__min_samples_leaf': [1, 2, ...], 'modelo__min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoos

In [16]:
print("Melhor F1:", round(busca_random_forest.best_score_, 3))

print("\nMelhores hiperparâmetros:")
for parametro, valor in busca_random_forest.best_params_.items():
    print(f"{parametro}: {valor}")

Melhor F1: 0.642

Melhores hiperparâmetros:
modelo__n_estimators: 200
modelo__min_samples_split: 20
modelo__min_samples_leaf: 2
modelo__max_features: log2
modelo__max_depth: None


In [17]:
melhor_random_forest = busca_random_forest.best_estimator_

resultado_random_forest_otimizada = cross_validate(
    melhor_random_forest,
    X_train,
    y_train,
    cv=validacao_cruzada,
    scoring=metricas,
    n_jobs=-1
)

print(
    "F1:",
    round(resultado_random_forest_otimizada["test_f1"].mean(), 3)
)

print(
    "Recall:",
    round(resultado_random_forest_otimizada["test_recall"].mean(), 3)
)

print(
    "Precision:",
    round(resultado_random_forest_otimizada["test_precision"].mean(), 3)
)

print(
    "ROC-AUC:",
    round(resultado_random_forest_otimizada["test_roc_auc"].mean(), 3)
)

F1: 0.642
Recall: 0.783
Precision: 0.545
ROC-AUC: 0.847


## Otimização e seleção do modelo

Na comparação inicial, a Regressão Logística apresentou melhor desempenho que a Random Forest em sua configuração inicial, principalmente em F1-score e Recall.

Antes da seleção definitiva do modelo, foi realizado um ajuste de hiperparâmetros da Random Forest utilizando RandomizedSearchCV.

A busca foi realizada exclusivamente sobre o conjunto de treinamento, utilizando validação cruzada estratificada com 5 folds e F1-score como métrica principal de otimização. Dessa forma, o conjunto de teste permaneceu isolado durante todo o processo de seleção e ajuste do modelo.

A Random Forest otimizada apresentou os seguintes resultados médios na validação cruzada:

- F1-score: 0.642
- Recall: 0.783
- Precision: 0.545
- ROC-AUC: 0.847

Para comparação, a Regressão Logística apresentou:

- F1-score: 0.633
- Recall: 0.803
- Precision: 0.522
- ROC-AUC: 0.846

Após a otimização, a Random Forest apresentou o melhor F1-score e maior Precision, mantendo ROC-AUC praticamente equivalente ao da Regressão Logística. A Regressão Logística permaneceu com Recall ligeiramente superior.

Considerando o F1-score como principal critério de seleção, a Random Forest otimizada foi escolhida como modelo candidato final, por apresentar o melhor equilíbrio entre Precision e Recall durante a validação cruzada.

### Próxima etapa

O modelo selecionado será agora avaliado no conjunto de teste, composto por 1.407 clientes que permaneceram isolados durante todo o processo de treinamento, comparação e ajuste de hiperparâmetros.

Essa avaliação permitirá estimar o desempenho do modelo em dados não utilizados durante sua seleção e verificar sua capacidade de generalização.

In [18]:
previsoes_teste = melhor_random_forest.predict(X_test)

probabilidades_teste = melhor_random_forest.predict_proba(X_test)[:, 1]

print("F1:", round(f1_score(y_test, previsoes_teste), 3))
print("Recall:", round(recall_score(y_test, previsoes_teste), 3))
print("Precision:", round(precision_score(y_test, previsoes_teste), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, probabilidades_teste), 3))

print("\nMatriz de confusão:")
print(confusion_matrix(y_test, previsoes_teste))

print("\nRelatório de classificação:")
print(classification_report(y_test, previsoes_teste))

F1: 0.624
Recall: 0.789
Precision: 0.517
ROC-AUC: 0.833

Matriz de confusão:
[[757 276]
 [ 79 295]]

Relatório de classificação:
              precision    recall  f1-score   support

           0       0.91      0.73      0.81      1033
           1       0.52      0.79      0.62       374

    accuracy                           0.75      1407
   macro avg       0.71      0.76      0.72      1407
weighted avg       0.80      0.75      0.76      1407



## Avaliação final no conjunto de teste

Após a seleção da Random Forest otimizada, o modelo foi avaliado no conjunto de teste, que permaneceu isolado durante as etapas de treinamento e ajuste de hiperparâmetros.

Os resultados obtidos foram:

- F1-score: 0.624
- Recall: 0.789
- Precision: 0.517
- ROC-AUC: 0.833
- Accuracy: 0.75

Dos 374 clientes que realmente apresentaram churn, o modelo identificou corretamente 295 e deixou de identificar 79.

Ao mesmo tempo, 276 clientes foram classificados como possíveis casos de churn, embora não tenham cancelado.

O Recall de 0.789 mostra que o modelo conseguiu identificar aproximadamente 79% dos clientes que efetivamente apresentaram churn.

A diferença relativamente pequena entre os resultados da validação cruzada e do conjunto de teste indica que o modelo manteve desempenho consistente em dados não utilizados durante o treinamento.

Do ponto de vista de negócio, o modelo pode ser utilizado para identificar clientes com maior risco de churn e direcionar ações preventivas de retenção.

In [19]:
#investigação dos 79 casos negativos
analise_erros = X_test.copy()

analise_erros["Churn_real"] = y_test
analise_erros["Churn_previsto"] = previsoes_teste
analise_erros["Probabilidade_churn"] = probabilidades_teste

falsos_negativos = analise_erros[
    (analise_erros["Churn_real"] == 1)
    & (analise_erros["Churn_previsto"] == 0)
].copy()

print("Quantidade de falsos negativos:", len(falsos_negativos))

print("\nProbabilidade de churn dos falsos negativos:")
print(
    falsos_negativos["Probabilidade_churn"]
    .describe()
    .round(3)
)

Quantidade de falsos negativos: 79

Probabilidade de churn dos falsos negativos:
count    79.000
mean      0.301
std       0.127
min       0.018
25%       0.226
50%       0.309
75%       0.393
max       0.499
Name: Probabilidade_churn, dtype: float64


In [ ]:
#comparação entre os 79 falso negativos e os 295 verdadeiro positivos nas três features 
# numéricas
verdadeiros_positivos = analise_erros[
    (analise_erros["Churn_real"] == 1)
    & (analise_erros["Churn_previsto"] == 1)
].copy()


print("Verdadeiros positivos:", len(verdadeiros_positivos))
print("Falsos negativos:", len(falsos_negativos))

comparacao_numericas = pd.DataFrame({
    "Verdadeiros positivos": verdadeiros_positivos[
        colunas_numericas
    ].mean(),
    "Falsos negativos": falsos_negativos[
        colunas_numericas
    ].mean()
})

comparacao_numericas["Diferença"] = (
    comparacao_numericas["Falsos negativos"]
    - comparacao_numericas["Verdadeiros positivos"]
)

comparacao_numericas.round(2)

Verdadeiros positivos: 295
Falsos negativos: 79


,Verdadeiros positivos,Falsos negativos,Diferença
tenure,12.90,32.44,19.54
MonthlyCharges,74.98,67.91,-7.07
TotalCharges,1123.00,2655.70,1532.70


### Análise dos falsos negativos

Os 79 clientes que apresentaram churn, mas não foram identificados pelo modelo, possuem um perfil diferente dos churners corretamente identificados.

Em média, os falsos negativos apresentam maior tempo de relacionamento (32,44 meses contra 12,90), menor mensalidade e maior valor total acumulado.

Esse resultado sugere que o modelo apresenta maior dificuldade para identificar churn em clientes com perfil aparentemente mais estável e maior tempo de relacionamento.

Na próxima etapa, serão analisadas as variáveis categóricas para verificar quais outras características podem estar associadas a esses erros.

In [21]:
#comparação dos falsos negativos com os verdadeiros negativos, nas variáveis
#categóricas

colunas_categoricas_analise = [
    "Contract",
    "InternetService",
    "PaymentMethod",
    "OnlineSecurity",
    "TechSupport"
]

for coluna in colunas_categoricas_analise:
    comparacao = pd.crosstab(
        analise_erros.loc[
            analise_erros["Churn_real"] == 1,
            "Churn_previsto"
        ].map({
            0: "Falso negativo",
            1: "Verdadeiro positivo"
        }),
        analise_erros.loc[
            analise_erros["Churn_real"] == 1,
            coluna
        ],
        normalize="index"
    ) * 100

    print(f"\n{coluna}")
    print(comparacao.round(1))


Contract
Contract             Month-to-month  One year  Two year
Churn_previsto                                         
Falso negativo                 44.3      45.6      10.1
Verdadeiro positivo            98.6       1.4       0.0

InternetService
InternetService       DSL  Fiber optic    No
Churn_previsto                              
Falso negativo       41.8         39.2  19.0
Verdadeiro positivo  19.7         77.3   3.1

PaymentMethod
PaymentMethod        Bank transfer (automatic)  Credit card (automatic)  \
Churn_previsto                                                            
Falso negativo                            30.4                     21.5   
Verdadeiro positivo                       12.2                      9.5   

PaymentMethod        Electronic check  Mailed check  
Churn_previsto                                       
Falso negativo                   24.1          24.1  
Verdadeiro positivo              63.1          15.3  

OnlineSecurity
OnlineSecurity       

### Perfil dos falsos negativos

A análise das variáveis categóricas reforçou a existência de um perfil menos típico entre os churners não identificados pelo modelo.

Enquanto os churners corretamente identificados estão fortemente associados a contrato mensal, fibra óptica, pagamento por electronic check e ausência de serviços de segurança e suporte, os falsos negativos apresentam com maior frequência contratos de longo prazo, pagamentos automáticos e serviços adicionais.

Em conjunto com o maior tempo médio de relacionamento observado anteriormente, esses resultados indicam que parte dos falsos negativos apresenta características normalmente associadas à permanência do cliente, dificultando sua identificação pelo modelo.

Ou seja, o modelo reconhece bem o padrão típico churner mas não prevê corretamente quando o cliente apresenta o padrão típico de não churn.

In [ ]:

from sklearn.model_selection import cross_val_predict

probabilidades_validacao = cross_val_predict(
    melhor_random_forest,
    X_train,
    y_train,
    cv=validacao_cruzada,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

print("Quantidade de previsões:", len(probabilidades_validacao))
print(
    "ROC-AUC:",
    round(roc_auc_score(y_train, probabilidades_validacao), 3)
)

Quantidade de previsões: 5625
ROC-AUC: 0.847


In [ ]:
#teste trheshold 1
thresholds = np.arange(0.20, 0.81, 0.05)

resultados_threshold = []

for threshold in thresholds:
    previsoes_threshold = (
        probabilidades_validacao >= threshold
    ).astype(int)

    resultados_threshold.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_train,
            previsoes_threshold
        ),
        "Recall": recall_score(
            y_train,
            previsoes_threshold
        ),
        "F1": f1_score(
            y_train,
            previsoes_threshold
        )
    })

comparacao_thresholds = pd.DataFrame(resultados_threshold)

comparacao_thresholds.round(3)

,Threshold,Precision,Recall,F1
0,0.20,0.381,0.956,0.545
1,0.25,0.407,0.937,0.568
2,0.30,0.433,0.912,0.588
3,0.35,0.466,0.888,0.612
4,0.40,0.489,0.855,0.622
5,0.45,0.515,0.823,0.634
6,0.50,0.544,0.783,0.642
7,0.55,0.567,0.718,0.633
8,0.60,0.595,0.657,0.624
9,0.65,0.633,0.585,0.608


In [26]:
#teste de threshold 2 (refinamento)

thresholds_refinados = np.arange(0.40, 0.61, 0.01)

resultados_threshold_refinado = []

for threshold in thresholds_refinados:
    previsoes_threshold = (
        probabilidades_validacao >= threshold
    ).astype(int)

    resultados_threshold_refinado.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_train,
            previsoes_threshold
        ),
        "Recall": recall_score(
            y_train,
            previsoes_threshold
        ),
        "F1": f1_score(
            y_train,
            previsoes_threshold
        )
    })

comparacao_thresholds_refinados = pd.DataFrame(
    resultados_threshold_refinado
)

comparacao_thresholds_refinados = (
    comparacao_thresholds_refinados
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)

comparacao_thresholds_refinados.round(3)

,Threshold,Precision,Recall,F1
0,0.50,0.544,0.783,0.642
1,0.51,0.550,0.771,0.642
2,0.49,0.540,0.792,0.642
3,0.48,0.534,0.800,0.641
4,0.52,0.553,0.760,0.640
5,0.47,0.529,0.809,0.640
6,0.46,0.523,0.817,0.637
7,0.53,0.555,0.743,0.636
8,0.45,0.515,0.823,0.634
9,0.54,0.559,0.730,0.634


## Análise do threshold de classificação

O threshold padrão de 0.50 foi inicialmente utilizado para converter as probabilidades estimadas pela Random Forest em classes de churn e não churn.

Para verificar se outro ponto de corte apresentaria melhor equilíbrio entre Precision e Recall, diferentes thresholds foram avaliados utilizando exclusivamente probabilidades obtidas por validação cruzada no conjunto de treinamento.

Na análise inicial, o maior F1-score foi observado próximo ao threshold de 0.50. Uma busca mais detalhada entre 0.40 e 0.60 confirmou que os thresholds 0.49, 0.50 e 0.51 apresentam resultados praticamente equivalentes, com F1-score próximo de 0.642.

Como o F1-score foi definido como principal métrica de seleção e não foi identificado ganho relevante com a alteração do ponto de corte, foi mantido o threshold padrão de 0.50.

A análise também demonstrou que thresholds menores aumentam o Recall, permitindo identificar uma proporção maior dos clientes que apresentam churn, porém com redução da Precision. Portanto, uma eventual alteração futura do threshold pode ser realizada de acordo com o custo e os objetivos das ações de retenção.

## Análise do threshold de classificação

O threshold padrão de 0.50 foi inicialmente utilizado para converter as probabilidades estimadas pela Random Forest em classes de churn e não churn.

Para verificar se outro ponto de corte apresentaria melhor equilíbrio entre Precision e Recall, diferentes thresholds foram avaliados utilizando exclusivamente probabilidades obtidas por validação cruzada no conjunto de treinamento.

Na análise inicial, o maior F1-score foi observado próximo ao threshold de 0.50. Uma busca mais detalhada entre 0.40 e 0.60 confirmou que os thresholds 0.49, 0.50 e 0.51 apresentam resultados praticamente equivalentes, com F1-score próximo de 0.642.

Como o F1-score foi definido como principal métrica de seleção e não foi identificado ganho relevante com a alteração do ponto de corte, foi mantido o threshold padrão de 0.50.

A análise também demonstrou que thresholds menores aumentam o Recall, permitindo identificar uma proporção maior dos clientes que apresentam churn, porém com redução da Precision. Portanto, uma eventual alteração futura do threshold pode ser realizada de acordo com o custo e os objetivos das ações de retenção.

In [27]:
import joblib
from pathlib import Path

diretorio_modelos = Path("../models")
diretorio_modelos.mkdir(parents=True, exist_ok=True)

caminho_modelo = diretorio_modelos / "random_forest_churn.joblib"

joblib.dump(
    melhor_random_forest,
    caminho_modelo
)

print("Modelo salvo em:", caminho_modelo)

Modelo salvo em: ..\models\random_forest_churn.joblib


In [28]:
modelo_carregado = joblib.load(
    "../models/random_forest_churn.joblib"
)

modelo_carregado

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessamento', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](19,)","['gender','SeniorCitizen','Partner',...,'PaymentMethod','MonthlyCharges', 'TotalCharges']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,19
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numericas', ...), ('categoricas', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specify

In [29]:
previsoes_modelo_carregado = modelo_carregado.predict(X_test)

previsoes_identicas = np.array_equal(
    previsoes_teste,
    previsoes_modelo_carregado
)

print(
    "As previsões são idênticas:",
    previsoes_identicas
)

As previsões são idênticas: True
